In [14]:
%load_ext autoreload
%autoreload 2 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
import sys 
import os
import pandas as pd 
import numpy as np

# This is a hack, to locate xpgaur.
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(project_root)

import xpgaur
from experiments.semantic_mutation.mutations import load_AutoEncoder_Gaur_model, load_AutoEncoder_Li_model, load_secureBERT_ae_model
from xpgaur.utils.traces_collector import get_traces_from_df

pd.set_option('display.max_colwidth', None)


In [16]:
queries = [
    "SELECT * FROM airport WHERE icao_code = 'LFOO'; select version() ;#'; ",
    "SELECT * FROM airport WHERE icao_code = 'LFOO'; sELECT version() ;#^!OKQ-UZL'; ",
]

df = pd.DataFrame({"full_query": queries, "label": [1] * len(queries)})

## SecureBERT representations

In [17]:

 # This is where my secureBERT and Li models were saved.
ANUBIS_PATH = "/home/infres/gquetel/repos/sqlia-dataset/models/output/models/"
xpgaur.ppths.add_model_path(ANUBIS_PATH)


sae_model = load_secureBERT_ae_model()
embeddings, _ = sae_model.preprocess_for_preds(df=df)
short_embeddings = [e[:15] for e in embeddings]
display(short_embeddings)



Some weights of RobertaModel were not initialized from the model checkpoint at ehsanaghaei/SecureBERT and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[array([ 0.05771367, -0.01670277, -0.03806906, -0.00770514, -0.01959788,
         0.03411995, -0.02104488, -0.02663971, -0.01650953, -0.00335604,
        -0.05036544,  0.02882844, -0.02231072,  0.08076227, -0.00339171],
       dtype=float32),
 array([ 0.05684033, -0.01615021, -0.03546479, -0.00768175, -0.01923623,
         0.03331474, -0.0212635 , -0.02736891, -0.01598713, -0.0041728 ,
        -0.04968273,  0.03062462, -0.0200344 ,  0.08074644, -0.00529755],
       dtype=float32)]

## GAUR Representation

In [18]:
# Configure GAUR trace mode, we use Mistral because this is one of the model for which
# we showed the semantic tags in a table (they are short). If we want to provide
# a representation of the features in the paper, this will be more coherent
# than displaying features out of nowhere.

trace_type = "mistral"
xpgaur.update_location_mysqlfiles(trace_type)


In [19]:


# We need a MySQL instrumented with Mistral running.
traces = get_traces_from_df(df=df, use_cache=False)
traces = pd.concat([traces, df], axis=1)
aegaurmistral_model = load_AutoEncoder_Gaur_model(trace_type=trace_type)

X_gaur, _ = aegaurmistral_model.preprocess_for_preds(df=traces)

# We only keep semantic information columns
cols_to_keep = [
    "Data Definition",
    "Data Import/Export",
    "Data Manipulation",
    "Data Query",
    "Database Management",
    "Locking & Concurrency",
    "Miscellaneous Operations",
    "Replication & Clustering",
    "Resource Management",
    "Security & Privileges",
    "Statement Control",
    "Stored Procedures & Functions",
    "System Information",
    "System Maintenance",
    "System Variables",
    "Temporary Objects",
    "Transaction Control",
    "Triggers & Events",
    "User Management",
    "Views",
]
X_gaur_filtered = X_gaur[cols_to_keep]
display(X_gaur_filtered)

100%|██████████| 2/2 [00:00<00:00, 203.05it/s]


,Data Definition,Data Import/Export,Data Manipulation,Data Query,Database Management,Locking & Concurrency,Miscellaneous Operations,Replication & Clustering,Resource Management,Security & Privileges,Statement Control,Stored Procedures & Functions,System Information,System Maintenance,System Variables,Temporary Objects,Transaction Control,Triggers & Events,User Management,Views
0,0,0,3,43,0,0,0,0,0,0,6,0,0,0,0,0,0,0,0,0
1,0,0,3,43,0,0,0,0,0,0,6,0,0,0,0,0,0,0,0,0


## Classification